# Medical Appointment No-Show Prediction

**Dataset**: KaggleV2-May-2016.csv (~110,527 appointments)

**Goal**: Predict if a patient will miss their appointment (`target = 1` = no-show)

This notebook is self-contained. Just place `KaggleV2-May-2016.csv` in the same folder.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from category_encoders import TargetEncoder

import xgboost as xgb

pd.set_option('display.max_columns', 50)
sns.set(style="whitegrid")
%matplotlib inline

## 1. Load the Data

In [2]:
df = pd.read_csv('KaggleV2-May-2016.csv', parse_dates=['ScheduledDay', 'AppointmentDay'])

print('Shape:', df.shape)
print('\nNo-show rate:')
print(df['No-show'].value_counts(normalize=True).round(4))

Shape: (110527, 14)

No-show rate:
No     0.7981
Yes    0.2019
Name: proportion, dtype: float64


## 2. Data Cleaning & Feature Engineering

In [3]:
# Rename columns
df = df.rename(columns={
    'PatientId': 'patient_id', 'AppointmentID': 'appointment_id',
    'Gender': 'gender', 'ScheduledDay': 'scheduled_day',
    'AppointmentDay': 'appointment_day', 'Age': 'age',
    'Neighbourhood': 'neighbourhood', 'Scholarship': 'scholarship',
    'Hipertension': 'hypertension', 'Diabetes': 'diabetes',
    'Alcoholism': 'alcoholism', 'Handcap': 'handicap',
    'SMS_received': 'sms_received', 'No-show': 'no_show'
})

# Target
df['target'] = df['no_show'].map({'Yes': 1, 'No': 0})
df = df.drop(columns=['no_show'])

# Waiting time (very predictive)
df['wait_days'] = (df['appointment_day'].dt.normalize() - df['scheduled_day'].dt.normalize()).dt.days
df['wait_days'] = df['wait_days'].clip(lower=0)

# Date features
df['scheduled_hour'] = df['scheduled_day'].dt.hour
df['appointment_dow'] = df['appointment_day'].dt.day_name()
df['appointment_is_weekend'] = df['appointment_day'].dt.weekday >= 5
df['same_day'] = (df['wait_days'] == 0).astype(int)

# Clean obviously wrong values
df = df[(df['age'] >= 0) & (df['age'] <= 115)]
df['handicap_binary'] = (df['handicap'] > 0).astype(int)

print('Cleaned shape:', df.shape)

Cleaned shape: (110526, 20)


## 3. Patient History Features (Very Powerful)

In [4]:
df = df.sort_values(['patient_id', 'scheduled_day'])

df['prev_appointments'] = df.groupby('patient_id').cumcount()
df['cum_no_shows']     = df.groupby('patient_id')['target'].cumsum().shift(1).fillna(0)
df['prev_no_show_rate'] = np.where(
    df['prev_appointments'] > 0,
    df['cum_no_shows'] / df['prev_appointments'],
    0.0
)
df['days_since_last'] = df.groupby('patient_id')['appointment_day'].diff().dt.days.fillna(999)

## 4. Preprocessor Definition

In [5]:
numeric = [
    'age', 'wait_days', 'scheduled_hour',
    'scholarship', 'hypertension', 'diabetes',
    'alcoholism', 'handicap_binary', 'sms_received',
    'prev_no_show_rate', 'days_since_last', 'same_day',
    'appointment_is_weekend'
]

categorical_high_card = ['neighbourhood']
categorical_low_card  = ['gender', 'appointment_dow']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric),
        ('high_card', TargetEncoder(), categorical_high_card),
        ('low_card', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), categorical_low_card)
    ],
    remainder='drop'
)

## 5. Model Pipeline & Training

In [6]:
X = df[numeric + categorical_high_card + categorical_low_card]
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=2025, stratify=y
)

pipe = Pipeline([
    ('prep', preprocessor),
    ('xgb', xgb.XGBClassifier(
        enable_categorical=False,           # we already encoded → no need
        n_estimators=1200,
        max_depth=6,
        learning_rate=0.035,
        subsample=0.82,
        colsample_bytree=0.78,
        eval_metric='aucpr',
        random_state=2025,
        scale_pos_weight = (y_train==0).sum() / (y_train==1).sum(),
        early_stopping_rounds=80
    ))
])

pipe.fit(
    X_train, y_train,
    xgb__eval_set=[(X_test, y_test)],
    xgb__verbose=100
)

## 6. Evaluation

In [ ]:
proba = pipe.predict_proba(X_test)[:, 1]
pred  = pipe.predict(X_test)

print('ROC AUC :', round(roc_auc_score(y_test, proba), 4))
print('PR AUC  :', round(average_precision_score(y_test, proba), 4))
print('\nClassification Report:\n', classification_report(y_test, pred))

## 7. Feature Importance

In [ ]:
# Get feature names after transformation
num_names   = numeric
high_names  = ['te_neighbourhood']
low_names   = pipe.named_steps['prep'].named_transformers_['low_card'].get_feature_names_out(categorical_low_card)
all_names   = num_names + high_names + list(low_names)

importances = pd.Series(
    pipe.named_steps['xgb'].feature_importances_,
    index=all_names
).sort_values(ascending=False)

plt.figure(figsize=(10, 8))
importances.head(20).plot(kind='barh', color='teal')
plt.title('Top 20 Most Important Features')
plt.gca().invert_yaxis()
plt.show()

## Done!

Next steps you may want to try:
- Use native categorical support (`enable_categorical=True` + leave strings as category dtype)
- Try LightGBM or CatBoost (excellent with categorical features)
- Hyperparameter tuning with Optuna / Hyperopt
- Add SHAP explanations
- Patient-grouped cross-validation (GroupKFold)